<a href="https://colab.research.google.com/github/mariajuliae/assistente_voz_gpt_whisper/blob/main/assistende_de_voz_whisper_gpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Defino o idioma

In [2]:
language = 'pt'

Gravação de áudio com Python +Js

In [4]:
# all imports
from IPython.display import Javascript, Audio
from google.colab import output
from base64 import b64decode
from io import BytesIO
!pip -q install pydub
from pydub import AudioSegment
import os

RECORD = """
const sleep  = time => new Promise(resolve => setTimeout(resolve, time))
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.onloadend = e => resolve(e.srcElement.result)
  reader.readAsDataURL(blob)
})
var record = time => new Promise(async resolve => {
  stream = await navigator.mediaDevices.getUserMedia({ audio: true })
  recorder = new MediaRecorder(stream)
  chunks = []
  recorder.ondataavailable = e => chunks.push(e.data)
  recorder.start()
  await sleep(time)
  recorder.onstop = async ()=>{
    blob = new Blob(chunks)
    text = await b2text(blob)
    resolve(text)
  }
  recorder.stop()
})
"""

def record(sec=5):
  display(Javascript(RECORD))
  js_result = output.eval_js('record(%d)' % (sec*1000))
  audio = b64decode(js_result.split(',')[1])
  file_name = 'request_audio.wav'
  with open(file_name, 'wb') as f:
    f.write(audio)
    return f'/content/{file_name}'

print("Ouvindo...\n")
record_file = record()
display(Audio(record_file, autoplay=True))

Ouvindo...



<IPython.core.display.Javascript object>

Reconhecimento de Fala com Whisper (OpenAI)

In [5]:
pip install git+https://github.com/openai/whisper.git -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [6]:
import whisper

model = whisper.load_model("small")
result = model.transcribe(record_file, fp16=False, language=language)
transcription = result["text"]
print(transcription)

 Testando meu assistente de voz.


Integração com a API do ChatGPT

In [7]:
!pip install openai

In [ ]:
from openai import OpenAI

client = OpenAI(
    api_key="API-KEY"
)

response = client.chat.completions.create(
    model="gpt-4",
    messages=[ { "role": "user", "content": transcription} ]
)

chatgpt_response = response.choices[0].message.content
print(chatgpt_response)

Sinterizando a Resposta do ChatGPT como VOZ (gTTS)

In [1]:
!pip install gTTs

In [ ]:
from gtts import gTTS

gtts_object = gTTS(text=chatgpt_response, lang=language, slow=False)
response_audio = "/content/response_audio.wav"
gtts_object.save(response_audio)
display(Audio(response_audio, autoplay=True))